# 05b - Model Monitoring: Service Orders (Tabular)

**Azure ML MLOps Workshop | Track B - Tabular Data**

### Context

In Notebook 05, you set up monitoring for the text classifier — but drift on
TF-IDF features is abstract and hard to visualize.

Tabular monitoring is where drift becomes **tangible**. You can see:
- "The distribution of equipment models shifted — we're seeing more EX100 orders"
- "Average quantity ordered increased by 40%"
- "A new JobCode code appeared that wasn't in training data"

### What you'll do:
1. Establish the training data baseline for the tabular model
2. Configure data drift, prediction drift, and data quality monitoring
3. Visualize what drift looks like on tabular features
4. Set up automated alerts

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Establish the Training Data Baseline

With tabular data, baselines are much richer than text — we can profile
distributions of every feature.

In [ ]:
import pandas as pd
import sys, os
sys.path.insert(0, os.path.abspath("../../src/track_b_tabular"))
from preprocess_os import load_and_clean_os

df = load_and_clean_os("../../data/service_orders_dataset.csv")

print("Training data baseline:")
print(f"  Total samples: {len(df):,}")
print(f"  Overhaul rate: {(df['label'] == 1).mean():.2%}")
print(f"  Preventive rate: {(df['label'] == 0).mean():.2%}")
print(f"\nTop 5 equipment models (baseline distribution):")
modelo_dist = df["EquipmentModel"].value_counts(normalize=True).head(5)
for modelo, pct in modelo_dist.items():
    print(f"  {modelo}: {pct:.1%}")
print(f"\nQtyOrdered baseline:")
print(f"  Mean: {df['QtyOrdered'].mean():.2f}")
print(f"  Median: {df['QtyOrdered'].median():.2f}")
print(f"  Std: {df['QtyOrdered'].std():.2f}")
print(f"\nJobCode distribution:")
jc_dist = df["JobCode"].value_counts(normalize=True).head(5)
for jc, pct in jc_dist.items():
    print(f"  {jc}: {pct:.1%}")

## Step 2: Configure Model Monitoring

Same monitoring API as Track A, but now the signals apply to
tabular features — which makes drift much more interpretable.

In [ ]:
from azure.ai.ml import Input
from azure.ai.ml.entities import (
    MonitoringTarget,
    MonitorDefinition,
    MonitorSchedule,
    RecurrencePattern,
    RecurrenceTrigger,
    ServerlessSparkCompute,
    DataDriftSignal,
    PredictionDriftSignal,
    DataQualitySignal,
    ReferenceData,
    DataDriftMetricThreshold,
    DataQualityMetricThreshold,
    PredictionDriftMetricThreshold,
    NumericalDriftMetrics,
    CategoricalDriftMetrics,
    DataQualityMetricsNumerical,
    DataQualityMetricsCategorical,
    AlertNotification,
)

ENDPOINT_NAME = "contoso-repair-classifier"  # <<<< CHANGE THIS - MUST MATCH THE NAME YOU USED IN NOTEBOOK 04b
DEPLOYMENT_NAME = "blue"

In [ ]:
reference_data = ReferenceData(
    input_data=Input(
        path="azureml:service-orders:2",
        type="uri_file",
    ),
    data_context="training",
)

monitoring_target = MonitoringTarget(
    ml_task="classification",
    endpoint_deployment_id=f"azureml:{ENDPOINT_NAME}:{DEPLOYMENT_NAME}",
)

In [ ]:
data_drift_signal = DataDriftSignal(
    reference_data=reference_data,
    metric_thresholds=DataDriftMetricThreshold(
        numerical=NumericalDriftMetrics(jensen_shannon_distance=0.25),
        categorical=CategoricalDriftMetrics(pearsons_chi_squared_test=0.3),
    ),
)

prediction_drift_signal = PredictionDriftSignal(
    metric_thresholds=PredictionDriftMetricThreshold(
        categorical=CategoricalDriftMetrics(pearsons_chi_squared_test=0.3),
    ),
)

data_quality_signal = DataQualitySignal(
    reference_data=reference_data,
    metric_thresholds=DataQualityMetricThreshold(
        numerical=DataQualityMetricsNumerical(null_value_rate=0.1),
        categorical=DataQualityMetricsCategorical(out_of_bounds_rate=0.1),
    ),
)

print("Monitoring signals configured:")
print("  1. Data drift (Jensen-Shannon: 0.25, Chi-squared: 0.3)")
print("  2. Prediction drift (Chi-squared: 0.3)")
print("  3. Data quality (null_value_rate: 0.1, out_of_bounds_rate: 0.1)")

In [ ]:
monitor_definition = MonitorDefinition(
    compute=ServerlessSparkCompute(
        instance_type="Standard_DS3_v2",
        runtime_version="3.3",
    ),
    monitoring_target=monitoring_target,
    monitoring_signals={
        "data_drift": data_drift_signal,
        "prediction_drift": prediction_drift_signal,
        "data_quality": data_quality_signal,
    },
    alert_notification=AlertNotification(
        emails=["your-email@example.com"],  # <<<< CHANGE THIS TO YOUR EMAIL ADDRESS
    ),
)

monitor_schedule = MonitorSchedule(
    name="contoso-repair-monitor",
    trigger=RecurrenceTrigger(
        frequency="week",
        interval=1,
        schedule=RecurrencePattern(hours=6, minutes=0, week_days=["Monday"]),
    ),
    create_monitor=monitor_definition,
)

created_monitor = ml_client.schedules.begin_create_or_update(monitor_schedule).result()
print(f"\nMonitor created: {created_monitor.name}")
print(f"  Schedule: Weekly on Mondays at 6:00 AM")
print(f"  Alert emails: <YOUR_EMAIL>")

## Step 3: Visualize Tabular Drift Scenarios

This is where tabular monitoring shines — drift is intuitive and visual.
Unlike text features (TF-IDF vectors), you can point at specific business changes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scenario 1: Equipment model distribution shift
ax = axes[0]
models = ["EX200", "DZ300", "LH400", "EX100", "Other"]
training_dist = [0.35, 0.20, 0.15, 0.12, 0.18]
drifted_dist = [0.15, 0.10, 0.30, 0.25, 0.20]
x = np.arange(len(models))
width = 0.35
ax.bar(x - width/2, training_dist, width, label="Training", color="#1f77b4")
ax.bar(x + width/2, drifted_dist, width, label="Production", color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45)
ax.set_title("Equipment Model Drift")
ax.set_ylabel("Proportion")
ax.legend()

# Scenario 2: Quantity ordered distribution shift
ax = axes[1]
np.random.seed(42)
train_qty = np.random.exponential(3, 5000)
prod_qty = np.random.exponential(6, 5000)
ax.hist(train_qty, bins=30, alpha=0.6, label="Training", color="#1f77b4", density=True)
ax.hist(prod_qty, bins=30, alpha=0.6, label="Production", color="#ff7f0e", density=True)
ax.set_title("QtyOrdered Drift")
ax.set_xlabel("Quantity")
ax.legend()

# Scenario 3: New categorical values
ax = axes[2]
jc_train = ["PM", "RP", "CM", "IN", "EM"]
jc_train_pct = [0.30, 0.25, 0.20, 0.10, 0.15]
jc_prod = ["PM", "RP", "CM", "IN", "NEW"]
jc_prod_pct = [0.20, 0.15, 0.10, 0.10, 0.45]
x = np.arange(len(jc_train))
ax.bar(x - width/2, jc_train_pct, width, label="Training", color="#1f77b4")
ax.bar(x + width/2, jc_prod_pct, width, label="Production", color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels(jc_prod)
ax.set_title("JobCode Drift (new code!)")
ax.set_ylabel("Proportion")
ax.legend()

plt.suptitle("Tabular Drift Scenarios — Easy to Interpret and Act On", fontsize=14)
plt.tight_layout()
plt.show()

print("\nCompare with Track A: TF-IDF drift is 'the word vector distribution shifted'")
print("Tabular drift is: 'we're seeing 2x more LH400 orders and a new job code appeared'")
print("Which one would you rather explain to a business stakeholder?")

## Step 4: View All Monitoring Schedules

In [ ]:
print("Active monitoring schedules:")
for schedule in ml_client.schedules.list():
    print(f"  {schedule.name} | Enabled: {schedule.is_enabled} | Trigger: {schedule.trigger.frequency}")

## Go to Azure ML Studio Now

Navigate to **Monitoring** and you should see **two monitors**:

| Monitor | Model | Drift is... |
|---------|-------|-------------|
| `contoso-lead-monitor` | Text classifier | Abstract (TF-IDF vector distributions) |
| `contoso-repair-monitor` | Tabular classifier | Concrete (equipment model %, quantity stats) |

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Tabular drift** | Concrete, business-interpretable drift signals |
| **Same monitoring API** | Identical SDK calls for text and tabular models |
| **Actionable alerts** | 'EquipmentModel distribution shifted' is actionable; 'TF-IDF vector moved' is not |
| **Data quality** | Catch null rates, new categorical values, out-of-range numerics |

---

**Next:** [06b - Pipeline Definition (Tabular)](./06b_pipeline_definition_tabular.ipynb)